Neural networks operate on numbers, so raw text must first be converted into a numerical representation. In many language models, text is split into tokens and converted into token IDs. An embedding layer then maps each token ID to a learned, continuous-valued vector that the network can process.

Embeddings can also represent sentences, paragraphs, or entire documents. In retrieval-augmented generation (RAG) systems that use vector search, embeddings represent text chunks, which may contain sentences, paragraphs, or larger sections. These embeddings help retrieve content relevant to a query.

Word2Vec is a family of methods for learning word embeddings. Its two main approaches are continuous bag-of-words (CBOW), which predicts a target word from its surrounding context, and skip-gram, which predicts surrounding words from a target word. These approaches draw on the idea that words appearing in similar contexts tend to have related meanings.

Word embeddings can have different numbers of dimensions. Higher-dimensional embeddings can potentially capture more complex relationships, but more dimensions do not automatically mean better embeddings. Their quality also depends on the training data, model, and task. Increasing dimensionality generally requires more storage and computation.

Embeddings are used in many real-world products, including recommendation systems. Netflix recommends shows you might end up binge-watching, while Amazon suggests products you might be interested in.

Behind the scenes, embeddings help represent users, content, and products numerically, allowing recommendation models to learn patterns and relationships. Vector search can help find relevant items, but embeddings are only one part of these systems, and using them does not necessarily require a dedicated vector database.

It can seem like magic when ChatGPT, powered by large language models (LLMs), interprets and responds to natural language. Under the hood, however, these models perform numerical computations on representations of text. Embeddings provide an initial representation, which the network processes to capture relationships and context.

Natural language is complex and full of nuance. Words can have different meanings and interpretations depending on their context. For example, “bank” can refer to a financial institution or the land alongside a river.

For a machine to process natural language, text must be encoded into a numerical representation. Modern text embedding models are trained to produce vectors that capture aspects of meaning, including how words are used in context.

Like LLMs, text embedding models learn from training data, often using large text datasets. Depending on the model, these datasets may include publicly available, licensed, proprietary, or synthetic content. The training process helps the models learn patterns and relationships that reflect meaning.

Embedding layers are a foundational component of many LLMs. Standalone text embedding models can also generate representations of sentences, paragraphs, or documents for tasks such as semantic search, clustering, and retrieval-augmented generation (RAG).

Much of application development revolves around adding, updating, deleting, and retrieving data. Traditional relational databases work well when retrieving data using explicit criteria, such as an ID, a category, or a date range. Many also support full-text search.

Vector search takes a different approach. For semantic text search, an embedding model converts both the stored text and the user’s query into numerical vectors. A database then compares these vectors to find similar items. This makes it possible to retrieve content based on similarity in meaning, even when the query and the content use different words.

Relation database has an inherent drawback where they rely on some form of Structured Query Language (SQL) to retrieve data from tables but the SQL is not well suited for natural language queries.

```sql
CREATE TABLE song (
    id SERIAL PRIMARY KEY,
    description TEXT NOT NULL
);
```

Traditional song search works well for exact fields such as title, artist, or genre, but struggles with vague descriptions, synonyms, and themes. Vector search can match a query such as “calm music for a rainy evening” to songs with similar descriptions even when they do not contain those exact words.

Embedding models turn text, images, or audio into numerical vectors that capture useful similarities. A vector database stores these vectors and finds items close to a query vector. For example, a search for “songs about starting over after a breakup” might find a song described as “finding hope after heartbreak,” even though the wording differs. The quality of the results depends on the embedding model, the song descriptions, and how the search is configured.

## Understanding Vector Embeddings

A vector embedding is a list of numbers that represents an item, such as a song description, in a way that helps a computer compare it with other items. A text embedding model can place descriptions with similar meanings close together, even when they use different words.

The code below loads the all-MiniLM-L6-v2 model and converts three song descriptions into embeddings. It then compares the first description with each one, including itself, and prints a similarity score. A higher score means the model considers the descriptions more similar.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "A song about finding hope after heartbreak.",
    "An uplifting track about starting over after a breakup.",
    "An energetic dance song for a summer party.",
]

embeddings = model.encode(sentences)
print(f"Dimensions: {len(embeddings[0])}")

idx_to_compare = 0

similarities = model.similarity(embeddings[idx_to_compare], embeddings)

print(sentences[idx_to_compare])

for idx, embedding in enumerate(embeddings):
  print(f"- {sentences[idx]}, Score: {similarities[idx_to_compare][idx]:.4f}")

Dimensions: 384
A song about finding hope after heartbreak.
- A song about finding hope after heartbreak., Score: 1.0000
- An uplifting track about starting over after a breakup., Score: 0.6546
- An energetic dance song for a summer party., Score: 0.3642


### Comparing song descriptions

The model creates a 384-dimensional embedding for each description. The code compares the first description with all three:

| Song description | Similarity score |
| --- | ---: |
| A song about finding hope after heartbreak. | 1.0000 |
| An uplifting track about starting over after a breakup. | 0.6546 |
| An energetic dance song for a summer party. | 0.3642 |

The first description scores 1.0000 because it is compared with itself. The other breakup description scores higher than the summer party description because its meaning is more similar. These scores show similarity between embeddings; they do not identify a song's genre.

This example shows how embeddings can help find song descriptions with similar meanings, even when they use different words.

## How Vector Database Retrieval Works

Vector databases store embeddings and index them for efficient similarity search. To search one, a model converts the query into a vector, and the database finds stored vectors that are similar to it. One common measure is cosine similarity, which compares the directions of two vectors. A higher score means their directions are more alike.

The fruit example makes this process easier to see. Each vector has three coordinates in the order **(sweetness, size, citrus intensity)**:

| Fruit | Vector | Description |
| --- | --- | --- |
| Banana | (8, 7, 0) | Very sweet, moderately large, non-citrus |
| Apple | (5, 6, 1) | Moderately sweet, medium-sized, slightly citrusy |
| Lemon | (2, 4, 8) | Not very sweet, smaller, strongly citrusy |

![Three-dimensional plot of banana, apple, lemon, and a query vector, with sweetness on the x-axis, size on the y-axis, and citrus intensity on the z-axis.](fruit-vectors-space.png)

The plot places apple and banana relatively close together because both have higher sweetness and size values and low citrus intensity. Lemon lies farther toward the citrus axis. This shows how the chosen features position fruits in a vector space. Cosine similarity compares directions from the origin, rather than straight-line distances on the plot.

In a real system, stored items and queries are encoded into the same embedding space. This example compares three hand-chosen vectors directly; a vector database indexes stored vectors so it can retrieve similar items efficiently at larger scale.

### Calculating cosine similarity

For two nonzero vectors $A$ and $B$, the formula is:

$$
\cos(\theta) = \frac{A \cdot B}{\lVert A \rVert\,\lVert B \rVert}
$$

The dot product multiplies matching coordinates and adds the results. The magnitudes in the denominator normalize vector length, so the score reflects direction. For the query $Q=(6,6,2)$ and apple $A=(5,6,1)$:

1. Dot product: $Q\cdot A=6(5)+6(6)+2(1)=30+36+2=68$.
2. Magnitudes: $\lVert Q \rVert=\sqrt{36+36+4}=\sqrt{76}$ and $\lVert A \rVert=\sqrt{25+36+1}=\sqrt{62}$.
3. Similarity: $68/(\sqrt{76}\sqrt{62})\approx0.9906$, displayed as **0.991** to three decimal places.

The next cell uses **(6, 6, 2)** as a query for a moderately sweet, medium-sized fruit with a little citrus and ranks the stored fruit vectors by cosine similarity.

In [ ]:
from math import sqrt

fruits = {
    'Banana': (8, 7, 0),
    'Apple': (5, 6, 1),
    'Lemon': (2, 4, 8),
}
query = (6, 6, 2)

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = sqrt(sum(x * x for x in a))
    magnitude_b = sqrt(sum(y * y for y in b))
    return dot_product / (magnitude_a * magnitude_b)

ranked = sorted(
    ((fruit, cosine_similarity(query, vector)) for fruit, vector in fruits.items()),
    key=lambda result: result[1],
    reverse=True,
)

for fruit, score in ranked:
    print(f'{fruit}: {score:.3f}')

The query scores about **0.991** with apple, **0.971** with banana, and **0.651** with lemon, so apple ranks first. These scores compare vector directions, not absolute differences in individual fruit features.

These coordinates were chosen by hand to describe fruit. They cannot meaningfully describe Apple Inc., because sweetness, size, and citrus intensity are fruit attributes. Real text embeddings are learned from data and have many more dimensions; individual coordinates usually do not have simple, human-readable meanings.

That's why vector database are so useful for RAG solution. 
Vector database use a nearest neighbor (NN) algorithm to determine similarity by calculating the distance between vector embeddings. When we query a vector database, it will return the k-nearest neighbors (k-NN), where k is the number of closet result. This mean the results of a search will be very close but aren't guaranteed to be exact. 

We need to keep this tradeoff of efficiency over absolute correctness in mind when choosing a vector database for your solution